In [0]:
# Load the source data into a DataFrame
taxi_df = spark.sql("SELECT * FROM users.jamin_solensky.taxi")

# Transformation: Filter trips with a distance greater than 2 miles
filtered_df = taxi_df.filter(taxi_df.trip_distance > 2)

# Transformation: Add a new column for trip duration in minutes
from pyspark.sql.functions import unix_timestamp, col

transformed_df = filtered_df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

# Aggregation: Calculate average fare amount and average trip duration by pickup zip code
aggregated_df = transformed_df.groupBy("pickup_zip").agg(
    {"fare_amount": "avg", "trip_duration_minutes": "avg"}
).withColumnRenamed("avg(fare_amount)", "avg_fare_amount") \
 .withColumnRenamed("avg(trip_duration_minutes)", "avg_trip_duration_minutes")

# Display the result
display(aggregated_df)